In [1]:
project_root_dir = !git rev-parse --show-toplevel
project_root_dir = project_root_dir[0]

import os
import sys
import itertools

import lark

sys.path.append(os.path.join(project_root_dir, 'lib'))
from hp_utils import *

In [2]:
list(
    map(
        lambda lr: hp_parse_learn_rate(lr),
        [
            '0.01',
            '0.05',
            '0.01,plateau()',
            '0.01,plateau(factor=1)',
            '0.01,plateau(patience=15)',
            '0.01,plateau(factor=3, patience=15)',
            '0.01,linear()',
            '0.01,linear(start_factor=0.8)',
            '0.01,linear(end_factor=0.9)',
            '0.01,linear(start_factor=0.8, end_factor=0.9)',
        ]
    )
)

[LearnRateParams(learn_rate=0.01, plateau=None, linear=None),
 LearnRateParams(learn_rate=0.05, plateau=None, linear=None),
 LearnRateParams(learn_rate=0.01, plateau=Plateau(factor=0.1, patience=10.0), linear=None),
 LearnRateParams(learn_rate=0.01, plateau=Plateau(factor=1.0, patience=10.0), linear=None),
 LearnRateParams(learn_rate=0.01, plateau=Plateau(factor=0.1, patience=15.0), linear=None),
 LearnRateParams(learn_rate=0.01, plateau=Plateau(factor=3.0, patience=15.0), linear=None),
 LearnRateParams(learn_rate=0.01, plateau=None, linear=Linear(start_factor=1.0, end_factor=0.0)),
 LearnRateParams(learn_rate=0.01, plateau=None, linear=Linear(start_factor=0.8, end_factor=0.0)),
 LearnRateParams(learn_rate=0.01, plateau=None, linear=Linear(start_factor=1.0, end_factor=0.9)),
 LearnRateParams(learn_rate=0.01, plateau=None, linear=Linear(start_factor=0.8, end_factor=0.9))]

In [3]:
# Old fashioned
expand_vars = dict(input_size=10, output_size=100)
units = [
    'LSTM(64)x10', 
    'LSTM(32)', 
    'Linear(200->ReLU)', 
    'Linear(dropout(0.5)->200->Tanh)',
    'Linear($input_size)',
    'Linear($output_size)',
    'Conv2d(conv(1->64(1)x9+bias)->TanH->gain->abs->LCN(9,2)->boxcar(9)+avg(2); from:s4_psd_01,397)',
]
hp_parse_model_units(units, expand_vars)

[StateModelUnitParams(module='LSTM', hidden_size=64, num_layers=10),
 StateModelUnitParams(module='LSTM', hidden_size=32, num_layers=1),
 LinearModelUnitParams(size=200, with_bias=True, nonlinearity=NonlinearityParams(module='ReLU', args=[], kwargs={}), dropout=None),
 LinearModelUnitParams(size=200, with_bias=True, nonlinearity=NonlinearityParams(module='Tanh', args=[], kwargs={}), dropout=OrdinaryParams(args=[0.5], kwargs={})),
 LinearModelUnitParams(size=10, with_bias=True, nonlinearity=None, dropout=None),
 LinearModelUnitParams(size=100, with_bias=True, nonlinearity=None, dropout=None),
 Conv2dModelUnitParams(module='Conv2d', convolution=Convolution(in_channels_count=1, out_channels_count=64, in_channels_count_per_kernel=1, kernel_size=9, with_bias=True, padding=None, stride=None), batch_norm2d=None, instance_norm2d=None, nonlinearity=NonlinearityParams(module='TanH', args=[], kwargs={}), with_gain=True, rectification='abs', normalization=Normalization(norm_method='LCN', lcn_windo

In [4]:
# New fashioned
expand_vars = dict(input_size=10, output_size=100)
units = [
    'LSTM: 64x10', 
    'LSTM: 32', 
    'Linear: 200->relu', 
    'Linear: dropout(0.5)->200->tanh',
    'Linear: dropout(0.5)->200-bias->tanh',
    'Linear: $input_size',
    'Linear: $output_size',
    'Conv2d: conv(1->64(1)x9+bias)->tanh->gain->abs->LCN(9,2)->boxcar(9)+avg(2); from:s4_psd_01,397',
    'Conv2d: conv(1->64(1)x9+bias,padding=1,stride=1)->Tanh->gain->abs->LCN(9,2)->boxcar(9)+avg(2); from:s4_psd_01,397',
    'ConvTranspose2d: conv(1->16(1)x3+bias,padding=1,stride=1)->LeakyReLU(0.2)',
]
hp_parse_model_units(units, expand_vars)

[StateModelUnitParams(module='LSTM', hidden_size=64, num_layers=10),
 StateModelUnitParams(module='LSTM', hidden_size=32, num_layers=1),
 LinearModelUnitParams(size=200, with_bias=True, nonlinearity=NonlinearityParams(module='relu', args=[], kwargs={}), dropout=None),
 LinearModelUnitParams(size=200, with_bias=True, nonlinearity=NonlinearityParams(module='tanh', args=[], kwargs={}), dropout=OrdinaryParams(args=[0.5], kwargs={})),
 LinearModelUnitParams(size=200, with_bias=False, nonlinearity=NonlinearityParams(module='tanh', args=[], kwargs={}), dropout=OrdinaryParams(args=[0.5], kwargs={})),
 LinearModelUnitParams(size=10, with_bias=True, nonlinearity=None, dropout=None),
 LinearModelUnitParams(size=100, with_bias=True, nonlinearity=None, dropout=None),
 Conv2dModelUnitParams(module='Conv2d', convolution=Convolution(in_channels_count=1, out_channels_count=64, in_channels_count_per_kernel=1, kernel_size=9, with_bias=True, padding=None, stride=None), batch_norm2d=None, instance_norm2d=N

In [5]:
# New fashioned
units = [
    'Conv2d: conv(1->64(1)x9+bias)->BatchNorm2d()',
    'Conv2d: conv(1->64(1)x9+bias)->BatchNorm2d(123, xyx=333)',
    'Conv2d: conv(1->64(1)x9+bias)->InstanceNorm2d()',
    'Conv2d: conv(1->64(1)x9+bias)->InstanceNorm2d(string_param="hello, world!")',
]
hp_parse_model_units(units)

[Conv2dModelUnitParams(module='Conv2d', convolution=Convolution(in_channels_count=1, out_channels_count=64, in_channels_count_per_kernel=1, kernel_size=9, with_bias=True, padding=None, stride=None), batch_norm2d=OrdinaryParams(args=[], kwargs={}), instance_norm2d=None, nonlinearity=None, with_gain=False, rectification=None, normalization=None, pooling=None, weights_source=None),
 Conv2dModelUnitParams(module='Conv2d', convolution=Convolution(in_channels_count=1, out_channels_count=64, in_channels_count_per_kernel=1, kernel_size=9, with_bias=True, padding=None, stride=None), batch_norm2d=OrdinaryParams(args=[123], kwargs={'xyx': 333}), instance_norm2d=None, nonlinearity=None, with_gain=False, rectification=None, normalization=None, pooling=None, weights_source=None),
 Conv2dModelUnitParams(module='Conv2d', convolution=Convolution(in_channels_count=1, out_channels_count=64, in_channels_count_per_kernel=1, kernel_size=9, with_bias=True, padding=None, stride=None), batch_norm2d=None, insta

In [6]:
hp_parse_artifact_source('15d_predict_noncausal_01:1')

ArtifactSourceParams(model_name='15d_predict_noncausal_01', model_version='1')

In [7]:
hp_parse_arg_list('42, param1=123, param2="Hello, world!"')

([42], {'param1': 123, 'param2': 'Hello, world!'})

In [12]:
list(
    map(
        lambda x: hp_parse_kwargs(x),
        [
            '',
            'param1=123, param2="Hello, world!"',
        ]
    )
)

[{}, {'param1': 123, 'param2': 'Hello, world!'}]

In [11]:
list(
    map(
        lambda x: hp_parse_universal_module(x),
        [
            'every',
            'every()',
            'every(200)',
            'every(200, 300)',
            'every(kwarg1=123)',
            'every(kwarg1=123, kwarg2=456)',
            'every(200, kwarg1=456)',
        ]
    )
)

[UniversalModuleParams(module_name='every', args=[], kwargs={}),
 UniversalModuleParams(module_name='every', args=[], kwargs={}),
 UniversalModuleParams(module_name='every', args=[200], kwargs={}),
 UniversalModuleParams(module_name='every', args=[200, 300], kwargs={}),
 UniversalModuleParams(module_name='every', args=[], kwargs={'kwarg1': 123}),
 UniversalModuleParams(module_name='every', args=[], kwargs={'kwarg1': 123, 'kwarg2': 456}),
 UniversalModuleParams(module_name='every', args=[200], kwargs={'kwarg1': 456})]